# Exercício 15 — AuditoriaGraph

**Entrega sem ecrã:** código executável abaixo + documentação em `docs/`.


## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)


## Classificação de risco e ramos

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
import os

llm = ChatGoogleGenerativeAI(
    model=(os.environ.get("GEMINI_MODEL") or "gemini-2.0-flash").replace("models/", ""),
    temperature=0.1,
)

achado = "Pagamentos duplicados a fornecedor sem dupla verificação em 3 meses."

ris_p = ChatPromptTemplate.from_messages([
    ("system", "Devolve só uma palavra: baixo | medio | alto"),
    ("human", "{achado}"),
])
risco = (ris_p | llm | StrOutputParser()).invoke({"achado": achado}).strip().lower()
print("Risco:", risco)

if "baixo" in risco:
    acao = "Gerar orientação breve ao gestor."
elif "medio" in risco:
    acao = "Solicitar evidências adicionais (extratos, reconciliações)."
else:
    acao = "Encaminhar para revisão humana prioritária."

rel_p = ChatPromptTemplate.from_messages([
    ("system", "Redige relatório curto PT-PT com risco, causa, consequência, recomendação."),
    ("human", "Achado:{a}\nDecisão:{d}", a=achado, d=acao),
])
print((rel_p | llm | StrOutputParser()).invoke({"a": achado, "d": acao}))


## Referências
- `docs/` desta pasta — explicações detalhadas.
- `app/` — espelho API opcional.
